# 🧠 FaceKey Studio Gen 3 — Kaggle Dual T4 GPU 3D Avatar Model Trainer
### Pure 3D Digital Human Animation: 52 ARKit Blendshapes + 3D Head Pose + 64-Band Log-Mel Acoustics

> [!IMPORTANT]
> **KAGGLE SETTINGS (RIGHT SIDEBAR):**
> 1. **Accelerator**: Select **GPU T4 x2** (Allocates 2x NVIDIA Tesla T4 GPUs with 30 GB combined VRAM)
> 2. **Internet**: Toggle **Internet ON** (Required to stream dataset chunks and sync model weights)

## 🌟 HOW TO RUN IN BACKGROUND WITH PC TURNED OFF (9 HOURS HEADLESS):
1. In the top right corner of Kaggle, click **Save Version**.
2. Under *Version Type*, select **Save & Run All (Commit)**.
3. Click **Save**.
4. **🎉 YOU CAN NOW SAFELY CLOSE YOUR BROWSER AND SHUT DOWN YOUR PC!**
   Kaggle runs the entire 100-epoch training run in the cloud. Every epoch automatically pushes `checkpoint_latest.pt`, `checkpoint_best.pt`, and `loss_curve.png` to Hugging Face Model Hub!

**🛡️ GEN 3 ARCHITECTURAL PILLARS:**
- **Pure 3D Avatar Targets**: Predicts 52 ARKit blendshapes + 3D head pose Euler angles + dental visibility.
- **Zero Landmark Bloat**: No 478 raw coordinates stored in dataset, making training 4x faster with minimal RAM usage.
- **True 64-Band Log-Mel Spectral Formants**: Full frequency coverage (20 Hz - 8000 Hz) captures distinct vowel and consonant shapes (/a/, /o/, /e/, /m/, /s/).
- **Expressive Paralinguistics & Tone**: 5-state emotion embedding (Neutral, Happy, Angry, Surprised, Sad) with continuous mood blending.
- **Auto-Resume & Fallback**: Automatically streams from Gen 3 dataset, falls back to Gen 2 if fresh, and auto-resumes weights.


## 💓 STEP 0: BROWSER KEEP-ALIVE HEARTBEAT (FOR INTERACTIVE SESSIONS)
> Run this cell if training interactively in your browser to prevent tab timeouts.


In [ ]:
# Browser Keep-Alive Heartbeat
from IPython.display import display, Javascript
display(Javascript('''
function keepAlive(){
    console.log('[FaceKey Gen3 Heartbeat] Session active at ' + new Date().toLocaleTimeString());
}
setInterval(keepAlive, 60000);
'''))
print('[+] Anti-sleep browser heartbeat initialized.')


## ⚡ STEP 1: GPU HARDWARE & VRAM VERIFICATION
> Confirms dual NVIDIA Tesla T4 allocation.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
import torch
print(f'[+] PyTorch Version: {torch.__version__}')
print(f'[+] CUDA Available: {torch.cuda.is_available()}')
print(f'[+] GPU Device Count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'    • GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)')


## 📦 STEP 2: CLONE GEN 3 CODEBASE & INSTALL DEPENDENCIES


In [ ]:
import os, shutil
APP_DIR = '/kaggle/working/FaceKeyAnimation'
if os.path.exists(APP_DIR):
    shutil.rmtree(APP_DIR)
!git clone -b Gen3 https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}
%cd {APP_DIR}
!pip install -q -U torch torchvision torchaudio huggingface_hub numpy scipy matplotlib
print('[+] Gen 3 Deep Learning Training Environment Ready on branch: Gen3')


## 🔑 STEP 3: HUGGING FACE AUTHENTICATION SETUP
> Uses Kaggle Secrets `HF_TOKEN` or falls back to local token file.


In [ ]:
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    print('[✓] Retrieved HF_TOKEN from Kaggle Secrets!')
except Exception:
    pass

if not hf_token and os.path.exists('/kaggle/working/hf_token.txt'):
    with open('/kaggle/working/hf_token.txt', 'r') as f:
        hf_token = f.read().strip()
    print('[✓] Retrieved HF_TOKEN from /kaggle/working/hf_token.txt')

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print(f'[✓] Hugging Face Hub Authenticated for automatic model sync!')
else:
    print('[!] Note: HF_TOKEN not found. Public datasets will stream freely; model checkpoints will save locally.')


## 🚀 STEP 4: LAUNCH GEN 3 3D AVATAR TRAINING (100 EPOCHS)
> Dual-GPU DataParallel training with Cosine Annealing, FP16 AMP, and phonetically-weighted viseme loss.


In [ ]:
!python scripts/train_speech_to_animation.py \
    --epochs 100 \
    --batch-size 1024 \
    --lr 5e-4 \
    --seq-len 48 \
    --stride 12 \
    --num-workers 4 \
    --checkpoints-dir /kaggle/working/checkpoints \
    --hf-repo VijayTheOne/facekey-gen3-dataset \
    --model-repo VijayTheOne/facekey-speech-to-animator


## 📊 STEP 5: VERIFY MODEL SIZE, LOSS CURVE & WEIGHT ARTIFACTS


In [ ]:
import os, glob
from IPython.display import Image, display
print('=' * 80)
print(' [✓] GEN 3 TRAINING COMPLETED SUCCESSFULLY!')
print('=' * 80)
ckpts = glob.glob('/kaggle/working/checkpoints/*.pt')
for c in sorted(ckpts):
    print(f'  • Checkpoint: {c} ({os.path.getsize(c) / (1024*1024):.2f} MB)')
curve = '/kaggle/working/checkpoints/loss_curve.png'
if os.path.exists(curve):
    print(f'\n[+] Displaying Loss Curve ({curve}):')
    display(Image(filename=curve))
